# AI Agent Workflow for Unit Compute

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(Path.cwd() / '.env')
print('Local environment loaded.')

In [ ]:
# Provider configuration: replace only this cell to switch providers.
# Examples: ChatGoogleGenerativeAI, ChatOpenAI, ChatBedrockConverse, or another LangChain chat model.
from langchain_google_genai import ChatGoogleGenerativeAI

chat_model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GOOGLE_API_KEY'],
    temperature=0,
)


## Step 1

Taking in user's request and translating it into a format understand by AI Agent

## Whats needed for future steps

What optional/not optional, instance types, etc. so we know if info is missing and can val;idate that the input from user is correct
Will create feedback loops based on required fields

In [ ]:
from typing import Literal, TypedDict

from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


class VMRequest(BaseModel):
    """The validated VM-request output for iteration 1."""

    operating_system: Literal['ubuntu', 'windows'] | None = None
    compute_size: Literal['small', 'medium', 'large'] | None = None
    region: str | None = None
    disk_gb: int | None = Field(default=None, ge=8, le=100)
    name: str | None = None
    purpose: str | None = None


# Policy configuration for iteration 1. Keep these mappings in backend code,
# not in the LLM prompt, so the model cannot choose arbitrary AWS values.
OPERATING_SYSTEM_TO_IMAGE_PARAMETER = {
    'ubuntu': {
        'label': 'Ubuntu Server 24.04 LTS',
        'image_ssm_parameter': (
            '/aws/service/canonical/ubuntu/server/24.04/stable/current/'
            'amd64/hvm/ebs-gp3/ami-id'
        ),
    },
    'windows': {
        'label': 'Windows Server 2022',
        'image_ssm_parameter': (
            '/aws/service/ami-windows-latest/'
            'Windows_Server-2022-English-Full-Base'
        ),
    },
}

COMPUTE_SIZE_TO_INSTANCE_TYPE = {
    'small': 't3.micro',
    'medium': 't3.small',
    'large': 't3.medium',
}


class RequestTranslator:
    """Uses any LangChain chat model to return a validated VMRequest."""

    SYSTEM_INSTRUCTION = (
        'Extract the requested VM configuration. Use null for missing details.'
    )

    def __init__(self, chat_model: BaseChatModel) -> None:
        self._structured_model = chat_model.with_structured_output(VMRequest)

    def translate(self, user_request: str) -> VMRequest:
        user_request = user_request.strip()
        if not user_request:
            raise ValueError('Please provide a VM request.')

        result = self._structured_model.invoke([
            SystemMessage(content=self.SYSTEM_INSTRUCTION),
            HumanMessage(content=user_request),
        ])
        return VMRequest.model_validate(result)


In [ ]:
# Preview the user-friendly-to-AWS mappings used by the future PlanBuilder.
{
    'operating_system': OPERATING_SYSTEM_TO_IMAGE_PARAMETER,
    'compute_size': COMPUTE_SIZE_TO_INSTANCE_TYPE,
}

In [ ]:
class TranslationState(TypedDict):
    user_request: str
    vm_request: VMRequest


def build_translation_graph(translator: RequestTranslator):
    
    def translate_request(state: TranslationState) -> dict:
        return {'vm_request': translator.translate(state['user_request'])}

    graph = StateGraph(TranslationState)
    graph.add_node('translate_request', translate_request)
    graph.add_edge(START, 'translate_request')
    graph.add_edge('translate_request', END)
    return graph.compile()

In [ ]:
translator = RequestTranslator(chat_model)
translation_graph = build_translation_graph(translator)

user_request = 'I need a small Ubuntu VM in us-east-1 for web testing.'
result = translation_graph.invoke({'user_request': user_request})
result['vm_request'].model_dump()